# 16.6 Batch 放不下怎麼辦？

# 第 16 章：由量測帶出效率 BKM

前置：已有第4章能forward與生成的模型。像整理廚房，先找哪一步最慢再改工作流程。CPU小例只驗證數值與資料契約；GPU速度、峯值記憶體與編譯收益必須在讀者的硬體實測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一次更新的生命週期**

忘記清零會累加；累積梯度時才刻意保留。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/backward.svg")))

**先想一想：**1token與10token的兩個micro-batch，各半權重正確嗎？

大batch放不下，拆成幾次算梯度、最後一次step。每個micro-batch先算loss sum，再除共同有效token總數，不能平均「各batch平均」。

**把直覺寫成數學：**gradient=Σ loss_sum_micro / total_valid_tokens。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import loss_sum

layer = nn.Linear(3, 4)
x = torch.randn(3, 3)
labels = torch.tensor([1, 2, 3])
total_tokens = 3
for rows in (slice(0, 1), slice(1, 3)):
    loss, count = loss_sum(layer(x[rows]), labels[rows])
    (loss / total_tokens).backward()
print("累積梯度", layer.weight.grad.norm().item())

**如何讀結果：**沒有step，驗證累加機制；真實訓練要先zero_grad、最後一次clip與step。

**只改一件事：**只改爲每micro獨自平均，比較與整batch的梯度差。
